# Lab 3 — Classification: Logistic Regression & Naive Bayes
**Machine Learning for the Natural Sciences**

In Week 1 we classified penguins with KNN (a distance-based method).
This week we learn two **parametric** classifiers that think very differently:

- **Logistic Regression** — learns a decision boundary (a line/plane in feature space)
- **Naive Bayes** — uses probability (Bayes' theorem) to assign the most likely class

Understanding how different classifiers "think" is essential before we move
to the powerhouse models (Random Forest, SVM) in Weeks 4–5.

---

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              ConfusionMatrixDisplay, classification_report)

RANDOM_STATE = 42

## 1. Prepare the Data

In [ ]:
df = sns.load_dataset("penguins").dropna()

feature_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = df[feature_cols]
y = df["species"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE
)

# Scale features — important for Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Training: {X_train_sc.shape[0]} | Test: {X_test_sc.shape[0]}")
print(f"Classes: {le.classes_}")

## 2. Logistic Regression
Despite the name, this is a **classification** algorithm. It learns a
linear decision boundary and outputs probabilities for each class.

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_sc, y_train)
y_pred_lr = log_reg.predict(X_test_sc)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Accuracy: {acc_lr:.3f}")

In [ ]:
# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(cm_lr, display_labels=le.classes_).plot(cmap="Blues")
plt.title("Logistic Regression")
plt.tight_layout()
plt.show()

In [ ]:
# The classification report gives precision, recall, and F1 per class
print("Classification Report — Logistic Regression:")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

### Logistic Regression: Under the Hood
The model learned **coefficients** for each feature, just like linear
regression — but instead of predicting a number, it runs the output
through a sigmoid function to get a probability.

In [ ]:
# Feature importance — which features drive the classification?
coef_df = pd.DataFrame(
    log_reg.coef_,
    columns=feature_cols,
    index=le.classes_
)
print("Logistic Regression Coefficients (per class):")
print(coef_df.round(3))

### 🔍 Your Turn
**TODO:** Look at the coefficients. For the Gentoo class, which feature
has the largest positive coefficient? What does that mean biologically?

*Your answer:*

## 3. Naive Bayes
Naive Bayes uses Bayes' theorem: P(class | features) ∝ P(features | class) × P(class)

It's "naive" because it assumes features are **independent** given the class.
That assumption is almost always wrong, but it works surprisingly well anyway!

In [ ]:
nb = GaussianNB()
nb.fit(X_train_sc, y_train)
y_pred_nb = nb.predict(X_test_sc)

acc_nb = accuracy_score(y_test, y_pred_nb)
print(f"Naive Bayes Accuracy: {acc_nb:.3f}")

In [ ]:
cm_nb = confusion_matrix(y_test, y_pred_nb)
ConfusionMatrixDisplay(cm_nb, display_labels=le.classes_).plot(cmap="Greens")
plt.title("Naive Bayes")
plt.tight_layout()
plt.show()

In [ ]:
# Probabilities — NB gives you confidence levels for each prediction
y_proba_nb = nb.predict_proba(X_test_sc)
proba_df = pd.DataFrame(y_proba_nb, columns=le.classes_)
print("First 5 predictions with probabilities:")
print(proba_df.head())

### 🔍 Your Turn
**TODO:** Look at the probability table above. Find a sample where the
model was uncertain (no class probability above 0.80). What were the
competing classes?

*Your answer:*

## 4. Head-to-Head Comparison
Let's compare all three classifiers we know so far: KNN, Logistic
Regression, and Naive Bayes.

In [ ]:
# KNN for comparison (using best k from Lab 1 — try 5 if you're not sure)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)
acc_knn = accuracy_score(y_test, y_pred_knn)

# Summary table
results = pd.DataFrame({
    "Model": ["KNN (k=5)", "Logistic Regression", "Naive Bayes"],
    "Accuracy": [acc_knn, acc_lr, acc_nb]
})
print(results.to_string(index=False))

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, name, y_p in zip(axes,
                          ["KNN", "Logistic Reg", "Naive Bayes"],
                          [y_pred_knn, y_pred_lr, y_pred_nb]):
    cm = confusion_matrix(y_test, y_p)
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap="Blues")
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 5. Cross-Validation
A single train/test split can be lucky or unlucky. **Cross-validation**
splits the data multiple ways and averages the results for a more
reliable estimate.

In [ ]:
models = {
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Naive Bayes": GaussianNB(),
}

print("5-Fold Cross-Validation Results:")
print(f"{'Model':<25} {'Mean Accuracy':>15} {'Std Dev':>10}")
print("-" * 52)

for name, model in models.items():
    scores = cross_val_score(model, scaler.fit_transform(X), y_encoded, cv=5)
    print(f"{name:<25} {scores.mean():>15.3f} {scores.std():>10.3f}")

### 🔍 Your Turn
**TODO:**
1. Which model wins in cross-validation? Is it the same one that won
   on the single test split?

   *Your answer:*

2. Why might cross-validation give different rankings than a single split?

   *Your answer:*

3. In one sentence each, describe how KNN, Logistic Regression, and
   Naive Bayes make predictions. What's fundamentally different about
   each approach?

   *Your answer:*

## 6. Decision Boundary Visualization (Bonus)
Using just 2 features, we can visualize how each classifier carves up
the feature space.

In [ ]:
from matplotlib.colors import ListedColormap

# Use just bill_length and bill_depth for 2D visualization
X_2d = df[["bill_length_mm", "bill_depth_mm"]].values
y_2d = le.fit_transform(df["species"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ListedColormap(["#FF9999", "#99FF99", "#9999FF"])

for ax, (name, model) in zip(axes, models.items()):
    model.fit(StandardScaler().fit_transform(X_2d), y_2d)

    # Create mesh grid
    sc = StandardScaler().fit(X_2d)
    X_2d_sc = sc.transform(X_2d)
    x_min, x_max = X_2d_sc[:, 0].min() - 0.5, X_2d_sc[:, 0].max() + 0.5
    y_min, y_max = X_2d_sc[:, 1].min() - 0.5, X_2d_sc[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap=colors)
    ax.scatter(X_2d_sc[:, 0], X_2d_sc[:, 1], c=y_2d, cmap=colors,
               edgecolors="k", linewidth=0.5, s=20)
    ax.set_title(name)
    ax.set_xlabel("Bill Length (scaled)")
    ax.set_ylabel("Bill Depth (scaled)")

plt.tight_layout()
plt.show()

---
## What to Submit
1. PDF with answers to all **🔍 Your Turn** sections
2. The comparison table and at least one confusion matrix figure
3. A brief paragraph: which classifier would you recommend for the
   Penguins dataset and why?

## What's Next
**Week 4** introduces **Random Forests** — an ensemble of decision trees
that often beats all three of this week's classifiers. You'll also start
**Data Adventure 2** on your chosen Data Path.